<div dir="rtl">

# HM_11 - מעבדה: לסדר קוד בלי לשבור אותו, ולקלוט קוד של מישהו אחר

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע (4261310) ולמסלול ידע ונתונים (תשפז_ב_4261110) · מאת ד״ר ולד לנדא וד״ר רן דובין · **מפגש 11:** ארכיטקטורה, קוד-בייס גדול ו-Code Handoff

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 11 למדנו את הכלל: **משנים מבנה רק כשהבדיקות ירוקות, ובצעדים קטנים; ובקוד של מישהו אחר - קודם מריצים, אחר כך קוראים.** במעבדה הזו לא תקראו על הכלל, תפעילו אותו: תקבלו סקריפט מבולגן (`messy_report.py`, דוח של "החנות"), תנעלו את הפלט שלו כ"אמת" (**golden master**), תסדרו אותו בארבעה-חמישה צעדים קטנים כשהבדיקה חייבת להישאר ירוקה, תמדדו מה קורה לקוד כשהנתונים גדלים, ובסוף תקלטו מאגר קוד זר ותכתבו עליו **מסמך העברה (handoff)**.

הכל רץ ב-Colab, בפייתון בלבד. (Git ו-commit אמיתיים אינם זמינים כאן, לכן "commit" במחברת הוא רישום בתוך הקוד: צעד נרשם רק אם הבדיקה ירוקה.)

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | רשת ביטחון: golden master | `golden_snapshot`, `is_green` | 6 דק' |
| ב' | Refactoring בצעדים קטנים | `low_stock_v1`, `revenue_by_category`, `top_products`, `add_revenue`, `low_stock_v2` + צעד עם עוזר AI | 12 דק' |
| ג' | קנה מידה: מודדים, לא מנחשים | `measure`, `totals_fast`, `same_answer` | 8 דק' |
| ד' | קליטת קוד זר ומסמך העברה | `inventory_repo`, `write_handoff` | 11 דק' |
| ה' | מסקנה | - | 2 דק' |

**מה חובה ומה מומלץ?** חובה (הליבה): חלק א' (שני שלבים), חלק ב' שלבים 1-3, חלק ג' (כל השלבים), חלק ד' שלבים 1-2. **מומלץ אם נשאר זמן, או להשלים בבית:** חלק ב' שלבים 4-5 (קבועים וצעד ה-AI) וחלק ד' שלב 3 (מסמך העברה אמיתי למאגר).

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` - **בעצמכם, בלי כלי AI**. המטרה היא להבין את התהליך, לא לקבל קוד. (רק בחלק ב' שלב 5 משתמשים בעוזר AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל מדידה או הרצה מפתיעה כותבים תחזית** בתא המיועד לכך. תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** בתאי ה"תשובה", במונחים של מה שלמדנו בהרצאה.

> **אל תיבהלו מהקוד.** כל פונקציה כאן היא 2-8 שורות, ובכל שלב מופיעים הסבר, דוגמה ורמז. אם נתקעתם: קראו שוב את ה-docstring (הטקסט בין שלוש המרכאות), ובקשו עזרה מבן הזוג.

## הגשה

בסוף: **Runtime ← Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל החלקים שעשיתם, ואז **File ← Download ← Download .ipynb** והגישו את הקובץ.

</div>

In [1]:
# תא הכנה - הריצו פעם אחת בתחילת העבודה
import os, re, sys, subprocess, random, time
import pandas as pd

print("✓ מוכן")

✓ מוכן


In [2]:
#@title תא הכנה - הנתונים (אפשר לא לפתוח) { display-mode: "form" }
import base64, gzip, pathlib
_FILES = {
    'messy_report.py': (
        'H4sIAJ8P2WoC/51UXW/TMBR976+48l4SKGFiKg8TfRijIERFq7ZjmqLI8mI3s5bYxnZK8+9xnI+mXUGCKA/+OPf6nnuOzQsltQVF'
        'BCUG3K/oiDdrpjKjEWVbKAgXQXg9AvddQC4J9UMJU4eONCMUp2YXIKkp0yZyYxR6hDpFKC1pmdohxpxiTKlUzo8TGfsCZWX6PEBc'
        'gGY7JkoGjxWkxLJM6qopM0ZuCyUugxv+LImw3FZu/qqel4JbrDRPGUo8vKhxUcF0xgIVx13JmFM0BtRlRkkyBimmw+2mkMzFF1Gm'
        'Zakeq+AQELZlRKYsgjAyrsN4R/KSmYCYlAnKRTb9THLD2t5pLmyAVrMfs+93M/j4ALc3m9mXxeqhPWgrNTyPYQdcQBZxywrTiXSI'
        'dwBXiaDBbgyX4TBz1zYrFXSy+CXrG9AT6AgKUrB/IRE9Oa2CyRGZzWIJE1iuFp/ubjfrMzzs//PI5S/wpvBzep5ErdLQAw0PeAtX'
        '7yc+bl/7rFXfNAp3fvTB3Zb9g/r7GFFvtf2Rc5KoICqgYbTleS5IcNmjtVQdnnpPukFeu9zygmFKKuNWX8NVs0NbkxIf4RakwE/u'
        '6jrMhz5bEqVSVW1jSIwy0hxButPe1MM+sk9IjvT0YWP4mzfni3tYbxa33+BmPltt1tcOnzMRkPCgLK9l1US4phXuFZn0iIHE2h/O'
        'c5nGPDkRXh/30dkgRrUV8b6etIieS9jZRLdcx/DurFv6R8bPq7qdB7f4/P7GusReB+1ucS0iI+LE9kcNuV+s1htY3y2X86+z1dDg'
        'gzZcvTB3FXFB2d6R78qvum409Y/4FrC/gRjDdAoI4/pNxhg1qZoHevQba6jBVswFAAA='
    ),
    'orders.csv': (
        '...'
    ),
    'products.csv': (
        'H4sIAJ8P2WoC/21Z227cyBF9D5B/mA9gCPa9+3Etx+vE66xgOTDytKBHtDTQaDiYSxzn6/dUNfvG8ZMk6DR5qvrUlcfT/HjdXv7Y'
        'PXaH8XXqtuNleppPP7rrYXf5YzufL/G342m3nbrz9Xjc76YT4H/9y/0wiO7tdH7Z/Da+HjdvxvNu2/3+7RsBZR98Z3ulu4fBMFRW'
        '0Pv99ZyQVvdBdUKYXhuAPYNVDT7NCat971wXXD+U5+oK+nF32CWssr3vjOxNANQy1HT/mi/T13l+2fxiWr4u9E6Aheo94SXjbYOv'
        'SYOIBXyAmZmIa9GFtRQE1ro3hbVvwDVv03vThV7JTDt0v077zf102DxMl5a2kuQK63rnk+/E0MAbV7s+hE5I2UuR4aKFF9pK9zp0'
        'BlToXhyDZQNuaNve6c6L3hATwWjVPVzG4346taSNIuu86oXMUJ2hNWHnew03K9MPIt2KMAVayOL1QfKF8EMVI21GNsKAh0Xnht7m'
        'R7qood9PT+Nh9/8bvkPvbOdDH1z2ml+faGgPPXERvTPpwkW4OVCRH3qEB3zHVjJeDmt8bQLJjyRbIgYX+uV5R4oaT4+bj+PpZW2F'
        'lb31UeFkuOZT8ienakuk6g2ku8QmX5VUPztTjBGOZCNtP2RJSv2TI7U9zvWGqVmfVC/pmnfblx8bCpTzyhbbWxIyPFBeYtsDTbgq'
        'ujvre6My3K3gxQKEibKgo3sCLVb7Fl6zh2cFZTBHYl0kJUN3Px5h5t1+dwT7+X+tBXqgcEHuKxJRw82Rxgbbi8XwJRqVuD3QWIE3'
        'COGrjIawW59YGeJCF5DURLoGpbqPM/L/fNoglg6PrRUC6cd1eEXIKQIZvz1Q2wAXKWjDUD5MlMwaX2UgSt4QkspguwI3QaEoc3pH'
        'wZrwbvnn5u553EFz4yp/xrSB0JYyX4O/PVPbYHyPYA4mJjpONLDp9khVtSgnLRfH2tPDLb65BxZgQPJTKVKhrH8/vPnb3eZu/Lqf'
        'FiM+T9vnTsFLnUWW9skCLRssk2eoGDgF4OkiUccbGixYM9SZXpGmB6rj6bm6wTJjBkPMKG1Ox5uKzzXdl91p2k9nxMp8PbeUuV7h'
        'FTbXCm3X+ELbKS62VI98ulntbvCJuiX5Qpk+5xIcW4EL94Dci2BXopcqkwndh+lHzFY17dAbR2aqXPTNUJCFMPLlQKHKNZxvHEm0'
        '4LKPETeIUFwglb/oNyMLsJCk5khTTrUyxRky2Zfp63Z8rRmikBiuVCYLBxez4Bp+qgss/fQ0k1HFjfAx7r/IHI9eQIWZ4wQgyYeL'
        '8+Ci++f5MDX5Ir5WklJtbIMiOd9gK4bch6GO+nwpSAUNNvFE1cArUaiy9uzQICs3InQB5QK1GA5h3c/fuVYeXmqyUvdeMuWcTeD8'
        'ClvISs+NkouNUqSgGmjiCpYODxXMNb5f18BCFT0aWgLqGXOJt6Z7//bjPza/PI7HS67uUZkwDK415IlFmda26CqcbCxvhn4sF2Hd'
        'Cp0YK35wb21m4VtkoWwEaQHVw+QODCnvN8Dm4+ZhP03/bZKAYyBFdRaYG1bwQhpBgkYXnaPIyka/vkJXkoDzJEpsxsoVtkpe6KVQ'
        'NmT0RuShujf763SZ58vz5uE4jS+tuxEVaJtF4A6SrxGV9vZElXgdpVG51EnOSaj7PzmRLCBfKm5nc8ZDVbs9UMyABGCAq+cXjEp/'
        'H09fr4/nRtiKijynqEUqyMUJV1H2PXoeZHaVKyOOZVxWtKPpzYqoU36tHzKsCr1Ab0XiZ3FEnOju5m/fJlzG9Wlh+GF32T5PBy6y'
        'lga9PLwhGCs080xgcEBFxj3yVMPJAmmjRoNtAqMJxPVS68jZj9XhdY1m0gmuXY8Q9aYMC+gxvowcf/PlkktxfrxB6x94zlq8hm67'
        'wTfUFdMga1U21K3wFXlIVZG6XR6cUOQadEMeV013qGi0TewRkdfD9rnqSDMaCcGRMFxITw9DhW6Ia+55HcdOAosaXLHWgcMFYnaJ'
        'Bowu2IYzVRxJI7PPoQKB3V0vl90BKqlqctZKoLKIKyo+R1ltTzTcDU1AIY5B8QVmDa/YU39lSeE297jBrvCNBYAi+wQd/cgJBcPH'
        '3fP0bfPhsPu2Vgx0i+YHnnTl+b6GN9z9wNsS3BXNwVG+sLuG156nESBEcVH2FsNQQ1ulS2qcMQIMC2sMsIjdcfNh+onOMXwq6slk'
        'bDUiXtb4Vi6BagPIpwFXIL3V4Iq05EEbGdCmCxJI/RW4ob30UKWpEIPp3s3z4+ZuPlzG3SGn7iqkkUzQMQ8mM7frI63TPXXwQmGs'
        'T3ckYM76SGWD0by4GAbaVyVefn1i7X+kPC/iO+IJCOhIY8KncfuyskLxdgHzS17loFmv4a3/DU04uLS8ghIIyRpdkad5TtELRKrM'
        'gpYqBdzwphWUolaNtyN8XdDb2935+We00fQ7z80Hr/DYleh+C7yhbTm/4O2xC5WMNjW61o0jIhiTB5dttBW2TTOGKpbUsRUUDMax'
        'cT8ivczf9+scI2gcci5OIPHZvoa3CcZSLkJL4VS2MTToVir4J+JfprZJyKEGt8lF0yqGNmIqxx0U8Pn5NH/f3O/2e/yI1N/PrxhW'
        'DSUvxA81vgtxvKeBM3VGQ+BUSF2f+kKBmajFgjhDhabFkOQdQ3qubrHMOz7X0SIYzZvPwSCpiu73m7v9nDXCWFoLwj7bp8WOkLaG'
        'FrZOpMnN2+wK12ATW5g0xAKUioqQvkZWXHnlSNehk+Jol7PbXq6nafPuNL5OjYPRIJGoh9R6CFrjNOjC2JjYeXtKQIvXaInTwrOL'
        'Ha3Q0B+H7DXa3zTgwhuCphZTxhaTHUcKucyn8YkZv+TdccQrHimHOMyynbSsb/GVNPQy4ww5EGnaWcETdcgTXZORZc8jqLtu0YU7'
        'BiLB1UHn60EncofhbT+t1Iysge4q7/SF8glXuApBt+3iTBpfHjIqUXQqrkN1DCS+Cz0kWOFmOHRQIajOLC+FQN7Oc71OSjJHtreq'
        'yoRaFmQhSJbi3YiePN0JrSpkIolSiCJohyiviNMFV1jSuBjI7dmB5MpnnizvrieqN40jHW1HUMPiBoYzFJqQ1YHKo/wRABooxV67'
        'G3i+fUUitCo2QHwBlDZbdBVxkrtUHxc20cjQvR8PT4A/tKpFwkZK0qK6NDPU2MIZLRK6TNxJjmMkrBqZnawpL+GZaSkg0IBWwIop'
        'x6JZ1ivsZvx2vx8PeNzcEEXBgEODjUTZKJTeAq3EwN+FREqlHOeQRQXNfvUkG3RoPq2lBZJ7ARai6CK85iHHFqjr3owYHT/jGvaN'
        'Ty3tn2B/7lcFqm2FrbIAt7bxe0jyVWigObws1QfaidssXTvU0Eq8vN+jEhuSA/D3f+ansYqwh+N8upyptIJsQJdkkwDQ/GcsU12g'
        '0JWgxtO5TEBVSDBdgPR5IybxdKtIXhnIPBckhhPaitrykUugP/nn9fW4+TQfp5aqdbFFdvESol22Qtdkre01t9PKpUvAqQpb0Y2f'
        'J4YhNnN8Cwjjgq0Zk1oCf4tS+XZt6N5eX79+nVD+zMtTy5p2xFQWdE4MbmjhNW1uuZBvQ+4YMGq26EI8KKp7gmSc0xQm2QZec4/5'
        '0cjYKvINoqH6NJ1358t42JK784IxHbFUBfVQ9vkCUbU+0jie+m1ac6eRWqBHuDlQjFA8q6Pd1VnXCIb1gUY0vGjF1drcDZIk+Z/t'
        'EqHYjehFI5snQkGfd5sDzSVY+qAp6DNzzooUy+2BYoLnXRoJU+cJyw8rfG2B1hT26LdMwYvu1x+0e17JByQwWNCklGcH3HmCNqxj'
        'J4saVMz0qkALX8ujYBhir84qg38SsCYa+4OYH+MDcZfXw4En9Wm/yiVaUdGkzxY+P9e2+Jqw8VHsXGqXq8ebWnxhTSOOo+KSv+QL'
        'FLgGXVNXPMIaXVUBBO67eXxFUO/LN/TydENqL+kiDA26YT7whQ+qbGtEEC28EKfvMvwxVqdPkwL6rcE1b4iP0HX1CDTaw0xaO+73'
        '5xVxThlwi805IOgWX1MXnLikiuuOyNys4JVSuDpQs1b6GqSmBt7oRdLqCNWkRD+tasbjTVahlSlSRW5oaEUzHhuqOi6KfK+zI0IE'
        'NVI29OF6+ZJDb5S0kxmPrRiodNGGYulz/wSL9kifpSMAAA=='
    ),
    'suppliers.csv': (
        'H4sIAJ8P2WoC/03MTQqDMBCG4b3gHXqAQRITrVm2atu9BwiBzCLgH/4svL2MBqbLeXjnW/d57gMuNngY3YDQo/N2CwNa744VpvE+'
        'FrdhmnRCQhc/Hi9QILKKNGd9g9TEhlyx11dtCmLN3MT8minYW5CC+py8ZP/cvdHkT/bvf1+x/+K+SpMTbHGZzusAAAA='
    ),
    'stock.csv': (
        'H4sIAJ8P2WoC/yWUS25cMQwE9zmLFmp9SPEWuYEReBbJxjaM5P556loRFKl6LbJnvr4/X//e/779ebXPj7ffvz5eP372rjZvGG3c'
        'MJtuWBxusqCWhNPWDeWaujsFRYMwW79hNbkHjIJiNl2ODimc0Q0fIhvcHNM9Y3G66QnTR3J42r6h3DmRM+WWOSx1TrdMHjU3IahB'
        'mafFDYhZUJZa3jA4nL6wlmELLSuoJbXjz65yy+5mbkaz0bJ50eZFe9OClg1lH6N3+evRzQz5MAbZJENLbA7DsIASjDfKtWS8iZaE'
        'kiw70ZLb9zLMTCgJJZnuQctByxnuPGg5ixpaDjs6UM7xF065s9BSaCmmW8ylli8UlGK6lWRsukxR7wQbRt3GU/em9djXtU0IWpKa'
        'd6QO5bHvrUlkgzAJUGQtUhCSC1DkTQvzavhFGt60hqerscjQMoIs6TyE8uFEy4Qy0TKh4F093lXeGKSImWCml6THvFfTAvOY915Y'
        'PGnxpMe8bgkyxCwoCzHbPwE95nXGeB/znhtsO20Gg3m10bKhYF5hXj3mrRugPOZ1DS2YV8F4Ay2YV8GLEi3JktKGUfrnqLTtlIw3'
        'mQvmVbIkzCvMK8yrg5YDBfPqbA4xzEk6Dy1QCkqJMAhQCkpBKeZSvKj8B6MyZfT71/Afq0sHK48FAAA='
    ),
    'legacy-repo/main.py': (
        'H4sIAJ8P2WoC/11Qy4rDMAy8+ytETjYsob0uuMf9iRCEqeViiO1gOaX9+yqPdpf1TaMZz2himkttwE9WoZYEU3GeKsQdXifkZZ6n'
        'SPVg8LXUmG9vyjqSUspTgLpknWLGDbOn/mS+FcjzAey/v7Q5NkO3sbtRKD70TvZPvUFf4B6R7dn8/iEEFlO8u2kh1odUiHyl7CWV'
        '/XET066o1JaaV4u/LhcLn4ij5I4BELNLhAjWQoeYnOyx26MnFtsg2ZuWjnpXb/fhPBoQ2UT5gxm4wBlIvEHO3pRlaSLdKuE90Cy9'
        'NS34MHSrowR/xxr7VpDbWqyO2dPjOMSoFwUwmeWhAQAA'
    ),
    'legacy-repo/loader.py': (
        'H4sIAJ8P2WoC/8vMLcgvKlEoSMxLSSxWAKKCFK60ovxcheT8vLTMdIVMiLyLY4hjvItnEBcXV0pqmkJOfmJKfHFpQUFOZmpRsYam'
        'FZcCEBSllpQW5QFN0CtKBconF5dppClVw7TW6sM16AGllDS5AHcYa2t8AAAA'
    ),
    'legacy-repo/scoring.py': (
        'H4sIAJ8P2WoC/0WOwQrCMBBE7/mKoadWIyLeBA8FexCsXgQFKSE0qQRMUrYp4t+bVmn3NPCYt9OQt6i9a8wTxraeAm7ichbXY1nw'
        'GE9FfuAo8/uYGGNKN+hqTzol/852DPFeWirRytjdY4MVrHEDfSQjCMZqoeSnS6rZlGE9WwcH6dCTA/neqXSagAVGkXc/DcmgkwrL'
        '/7KIp98c24x9AT70rU7PAAAA'
    ),
    'legacy-repo/config.py': (
        'H4sIAJ8P2WoC/3NxDHGMd/EMUrBVUNJT4gqP9/eLD/H0dQXyDfTMgHwfV0cXMMeEy9cxAsY1MuQCADBGZk86AAAA'
    ),
    'legacy-repo/README.md': (
        'H4sIAJ8P2WoC/1NWKE7OL0pNTixK4QoqzVPITczM0yuo5AIA58z/UBgAAAA='
    ),
    'legacy-repo/suppliers.csv': (
        'H4sIAJ8P2WoC/03MTQqDMBCG4b3gHXqAQRITrVm2atu9BwiBzCLgH/4svL2MBqbLeXjnW/d57gMuNngY3YDQo/N2CwNa744VpvE+'
        'FrdhmnRCQhc/Hi9QILKKNGd9g9TEhlyx11dtCmLN3MT8minYW5CC+py8ZP/cvdHkT/bvf1+x/+K+SpMTbHGZzusAAAA='
    ),
}
for _name, _b64 in _FILES.items():
    pathlib.Path(_name).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_name).write_bytes(gzip.decompress(base64.b64decode(_b64)))
print('✓ הקבצים נוצרו:', ', '.join(_FILES))

✓ הקבצים נוצרו: messy_report.py, orders.csv, products.csv, suppliers.csv, stock.csv, legacy-repo/main.py, legacy-repo/loader.py, legacy-repo/scoring.py, legacy-repo/config.py, legacy-repo/README.md, legacy-repo/suppliers.csv


<div dir="rtl">

---

# חלק א' - רשת ביטחון: golden master (6 דק')

## הרקע

בהרצאה: **Refactoring = שינוי מבנה בלי לשנות התנהגות.** כדי לדעת שההתנהגות לא השתנתה צריך בדיקה. אבל ל"קוד בן חצי שנה" אין בדיקות. הפתרון הזמני: **golden master** - מריצים את התוכנית כמו שהיא היום, שומרים את הפלט כ"אמת", ואחרי כל שינוי בודקים שהפלט **זהה אות באות**. זה לא בודק שהקוד *נכון*, אלא שלא *שברנו* אותו תוך כדי סידור.

## הקוד המבולגן

הקובץ `messy_report.py` נוצר בתא ההכנה. זה דוח של "החנות": הכנסות לפי קטגוריה, 5 מוצרים מובילים, התראות מלאי נמוך וספקים גרועים. קראו אותו:

</div>

In [3]:
print(open("messy_report.py", encoding="utf-8").read())

import pandas as pd
import sys

def main():
    # load
    o = pd.read_csv("orders.csv")
    p = pd.read_csv("products.csv")
    s = pd.read_csv("suppliers.csv")
    st = pd.read_csv("stock.csv")
    # revenue by category
    o["rev"] = o["quantity"] * o["unit_price"]
    m = o.merge(p[["product_id", "category"]], on="product_id")
    g = m.groupby("category")["rev"].sum().sort_values(ascending=False)
    print("REVENUE BY CATEGORY")
    for k, v in g.items():
        print(k, round(v, 0))
    print()
    # top products
    t = o.groupby("product_name")["rev"].sum().sort_values(ascending=False).head(5)
    print("TOP 5 PRODUCTS")
    for k, v in t.items():
        print(k, round(v, 0))
    print()
    # low stock
    d = o.groupby("product_id")["quantity"].sum() / 365
    x = p.merge(s, on="supplier_id").merge(st, on="product_id")
    x["d"] = x["product_id"].map(d).fillna(0)
    x["rop"] = x["d"] * x["lead_time_days"] + 3 * x["d"]
    a = x[x["on_hand"] < x["rop"]].copy()
    a["gap"]

<div dir="rtl">

**(א)** סמנו לפחות **שלושה "ריחות קוד"** בקובץ (מהשקף: פונקציה ארוכה, שכפול, מספר קסם, שם לא ברור, קובץ אחד לכל הדברים). ציינו שורה או שם מהקוד.

**תשובה:** שמות לא ברורים: x, d, a, st, rop. מספרי קסם: 365 ו-3 בנוסחת נקודת ההזמנה, ואין שם שאומר אם 3 הוא ימי ביטחון. שכפול: שתי לולאות הדפסה זהות (`for k, v in ... print(k, round(v, 0))`). ופונקציה אחת (main) שעושה הכל: טוענת, מחשבת ומדפיסה.

</div>

<div dir="rtl">

## שלב 1 - שומרים את ה"אמת" (חובה)

ממשו `golden_snapshot(script, path)`: מריצה את הסקריפט `script` כתוכנית פייתון נפרדת, **שומרת** את מה שהוא הדפיס (stdout) בקובץ `path`, ומחזירה את הטקסט.

**כלים:** `subprocess.run([sys.executable, script], capture_output=True, text=True)` מריצה סקריפט, ו-`result.stdout` הוא מה שהודפס. (`sys.executable` הוא פייתון שמריץ את המחברת.) כתיבה לקובץ: `open(path, "w", encoding="utf-8")` ו-`f.write(text)`.

</div>

In [4]:
def golden_snapshot(script, path="expected_output.txt"):
    """Runs `script` (a .py file) as a separate python process, saves what it printed (stdout) into `path`
    and returns that text.
    e.g. a script containing print("hi") -> returns "hi\n" and the file `path` contains "hi\n"
    """
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(result.stdout)
    return result.stdout

In [5]:
# בדיקה - חלק א', שלב 1
def check_a1():
    with open("_tiny.py", "w", encoding="utf-8") as f:
        f.write('print("hello")\nprint(1 + 2)\n')
    out = golden_snapshot("_tiny.py", "_tiny_expected.txt")
    assert out == "hello\n3\n", "golden_snapshot: צריך להחזיר את הטקסט שהסקריפט הדפיס (stdout)"
    assert open("_tiny_expected.txt", encoding="utf-8").read() == "hello\n3\n", "golden_snapshot: צריך גם לשמור את הטקסט בקובץ path"
    with open("_tiny.py", "w", encoding="utf-8") as f:
        f.write('for i in range(3):\n    print("row", i)\n')
    assert golden_snapshot("_tiny.py", "_tiny_expected.txt") == "row 0\nrow 1\nrow 2\n", "golden_snapshot: צריך להריץ את הסקריפט בכל קריאה מחדש"
    return True

check_a1() and print("✓ חלק א' שלב 1: golden_snapshot עובדת")

✓ חלק א' שלב 1: golden_snapshot עובדת


<div dir="rtl">

עכשיו מצלמים את ה"אמת" של הדוח המבולגן. הקובץ `expected_output.txt` ישמש אותנו בכל חלק ב':

</div>

In [6]:
golden_snapshot("messy_report.py")
golden = open("expected_output.txt", encoding="utf-8").read()
print(golden)

REVENUE BY CATEGORY
Office 185100.0
Home 182963.0
Tech 107257.0
Sports 105490.0
Kitchen 105468.0

TOP 5 PRODUCTS
Plant Pot Plus 33596.0
Desk Organizer Basic 33379.0
Stapler Plus 20522.0
Office Chair Mat Plus 18829.0
Storage Basket Pro 15467.0

LOW STOCK ALERTS: 57
P039 Office Chair Mat Pro 1 12.16
P154 Plant Pot Plus 5 12.25
P017 Desk Organizer Basic 12 17.28
P088 Water Bottle Mini 1 6.19
P160 Bath Towel Mini 0 3.31

WORST SUPPLIERS
Supplier A 0.8
Supplier H 0.83
Supplier D 0.88



<div dir="rtl">

## שלב 2 - האם עדיין ירוק? (חובה)

ממשו `is_green(script, path)`: מריצה את הסקריפט שוב, ומחזירה `True` אם הפלט **זהה בדיוק** לטקסט השמור בקובץ `path`, אחרת `False`.

**כלים:** אותה הרצה כמו בשלב 1. קריאת קובץ: `open(path, encoding="utf-8").read()`. השוואה מדויקת: `==`. סקריפט שקורס לא מדפיס כלום, ולכן ייחשב אדום.

</div>

In [7]:
def is_green(script, path="expected_output.txt"):
    """Runs `script` again and returns True if its output is IDENTICAL to the text saved in `path`.
    e.g. is_green("messy_report.py") -> True (as long as nothing changed)
    """
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    saved = open(path, encoding="utf-8").read()
    return result.stdout == saved

In [8]:
# בדיקה - חלק א', שלב 2
def check_a2():
    with open("_tiny.py", "w", encoding="utf-8") as f:
        f.write('print("hello")\nprint(1 + 2)\n')
    golden_snapshot("_tiny.py", "_tiny_expected.txt")
    assert is_green("_tiny.py", "_tiny_expected.txt") is True, "is_green: סקריפט שלא השתנה צריך להיות ירוק (True)"
    with open("_tiny.py", "w", encoding="utf-8") as f:
        f.write('print("hello")\nprint(1 + 3)\n')
    assert is_green("_tiny.py", "_tiny_expected.txt") is False, "is_green: הפלט השתנה (4 במקום 3) - צריך להיות אדום (False)"
    with open("_tiny.py", "w", encoding="utf-8") as f:
        f.write('raise SystemExit(1)\n')
    assert is_green("_tiny.py", "_tiny_expected.txt") is False, "is_green: סקריפט שקורס לא מדפיס כלום - צריך להיות אדום"
    assert is_green("messy_report.py") is True, "is_green: messy_report.py לא שונה, אז מול expected_output.txt הוא צריך להיות ירוק"
    return True

check_a2() and print("✓ חלק א' שלב 2: is_green עובדת, והדוח המבולגן ירוק")

✓ חלק א' שלב 2: is_green עובדת, והדוח המבולגן ירוק


<div dir="rtl">

## מה בודק ה-golden ומה לא?

הנה ארבעה "שיפורים" לקובץ. הקוד בתא הבא יוצר מכל אחד קובץ זמני `_variant.py` ומריץ עליו את `is_green`.

</div>

<div dir="rtl">

**לפני שמריצים:** איזה מהארבעה ייתן **ירוק** ואיזה **אדום**? (קראו את התיאורים בתא הבא.)

**התחזית שלי:** א (שינוי שם עמודה בכל המקומות) וב (הערות ושורות ריקות) משנים רק מבנה ולכן ירוקים. ג (3 ימי ביטחון ← 2) משנה את החישוב ולכן אדום. ד (עיגול ל-2 ספרות) משנה את הפלט ולכן אדום.

</div>

In [9]:
src = open("messy_report.py", encoding="utf-8").read()
variants = {
    "א. שינוי שם העמודה rop ← reorder_point (בכל המקומות)": src.replace('"rop"', '"reorder_point"'),
    "ב. הוספת הערות ושורות ריקות": "# הערה\n\n" + src.replace("    # load", "    # --- load ---\n"),
    "ג. ימי ביטחון: 3 ← 2": src.replace('3 * x["d"]', '2 * x["d"]'),
    "ד. עיגול הכנסות ל-2 ספרות במקום 0": src.replace("round(v, 0)", "round(v, 2)"),
}
for name, text in variants.items():
    with open("_variant.py", "w", encoding="utf-8") as f:
        f.write(text)
    print("ירוק ✓" if is_green("_variant.py") else "אדום ✗", "-", name)

ירוק ✓ - א. שינוי שם העמודה rop ← reorder_point (בכל המקומות)


ירוק ✓ - ב. הוספת הערות ושורות ריקות


אדום ✗ - ג. ימי ביטחון: 3 ← 2


אדום ✗ - ד. עיגול הכנסות ל-2 ספרות במקום 0


<div dir="rtl">

**(ב)** הבדיקה נשארה ירוקה אחרי שינוי **א**. האם זה אומר שהדוח **נכון**? ומה הייתה אומרת הבדיקה אילו היה בקוד המקורי באג (למשל נוסחה שגויה)?

**תשובה:** לא. golden מוכיח רק שהפלט זהה למה שהיה (צילום מצב), ולא שהוא נכון. באג שקיים בקוד המקורי נשמר בפלט ה'זהב', ולכן שינוי מבנה שמשאיר אותו ייחשב ירוק. זה בסדר, כי המטרה ב-Refactoring היא התנהגות זהה, ולא תיקון.

</div>

<div dir="rtl">

**(ג)** מה יקרה לבדיקה אם הדוח היה מדפיס גם **את התאריך של היום** (או מספר אקראי)? איך אפשר לפתור?

**תשובה:** הפלט ישתנה בכל הרצה, ולכן הבדיקה תהיה אדומה תמיד גם אם לא שינינו כלום. פותרים אם מסירים או מקפיאים את החלק המשתנה (למשל מזריקים תאריך קבוע, או מסננים את השורה לפני ההשוואה).

</div>

<div dir="rtl">

---

# חלק ב' - Refactoring בצעדים קטנים (12 דק')

## הרקע

המעגל מההרצאה: **בדיקות ירוקות ← צעד ממוקד אחד ← קוראים מה השתנה ← בדיקות ירוקות? ← commit.** אדום? **חוזרים צעד אחורה** (לא מתקנים על גבי תקלה). הדוח המבולגן מתפרק לחלקים; בכל צעד אתם כותבים **גרסה מסודרת של פונקציה אחת**, ו"המנוע" (תא מוסתר למטה) מרכיב את כל התוכנית מהחלקים שלכם ובודק שהפלט זהה ל-`expected_output.txt` מחלק א'. אם ירוק - הצעד נרשם ברשימת ה-commits. אם אדום - הוא לא נשמר ואתם חוזרים לגרסה הקודמת.

הפונקציה הראשונה כבר חולצה על ידי "מישהו" מתוך הדוח, אבל נשארה עם השמות המבולגנים:

</div>

In [10]:
def low_stock_messy(o, p, s, st):
    """The low-stock block of messy_report.py, extracted as is (names unchanged)."""
    d = o.groupby("product_id")["quantity"].sum() / 365
    x = p.merge(s, on="supplier_id").merge(st, on="product_id")
    x["d"] = x["product_id"].map(d).fillna(0)
    x["rop"] = x["d"] * x["lead_time_days"] + 3 * x["d"]
    a = x[x["on_hand"] < x["rop"]].copy()
    a["gap"] = a["rop"] - a["on_hand"]
    return a.sort_values("gap", ascending=False)

In [11]:
#@title תא מנוע - מרכיב את התוכנית מהחלקים שלכם ובודק מול ה-golden (אפשר לא לפתוח) { display-mode: "form" }
import io, ast, inspect, difflib, textwrap, contextlib

COMMITS = {}   # "commit log": step name -> True (נרשם רק כשה-golden ירוק)


def _load():
    return [pd.read_csv(f"{name}.csv") for name in ("orders", "products", "suppliers", "stock")]


def _default_by_category(orders, products):
    o = orders.copy()
    o["rev"] = o["quantity"] * o["unit_price"]
    m = o.merge(products[["product_id", "category"]], on="product_id")
    return m.groupby("category")["rev"].sum().sort_values(ascending=False)


def _default_top(orders, n=5):
    o = orders.copy()
    o["rev"] = o["quantity"] * o["unit_price"]
    return o.groupby("product_name")["rev"].sum().sort_values(ascending=False).head(n)


def run_report(parts=None):
    """Runs the whole report. `parts` may replace: alerts, by_category, top. Returns the printed text."""
    parts = parts or {}
    o, p, s, st = _load()
    by_category = parts.get("by_category") or _default_by_category
    top = parts.get("top") or _default_top
    alerts_fn = parts.get("alerts") or low_stock_messy
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        print("REVENUE BY CATEGORY")
        for k, v in by_category(o.copy(), p).items():
            print(k, round(v, 0))
        print()
        print("TOP 5 PRODUCTS")
        for k, v in top(o.copy(), 5).items():
            print(k, round(v, 0))
        print()
        a = alerts_fn(o.copy(), p, s, st)
        print("LOW STOCK ALERTS:", len(a))
        for i in range(min(5, len(a))):
            r = a.iloc[i]
            print(r["product_id"], r["name_x"], int(r["on_hand"]), round(r["rop"], 2))
        print()
        x = p.merge(s, on="supplier_id").merge(st, on="product_id")
        y = x.groupby("name_y")["on_time_rate"].mean().sort_values()
        print("WORST SUPPLIERS")
        for i in range(3):
            print(y.index[i], round(y.iloc[i], 2))
    return buf.getvalue()


def refactor_step_check(message, quiet=False, **parts):
    """The safety net: runs the report with the given parts and compares with expected_output.txt.
    Green -> the step is recorded in COMMITS. Red -> prints the first differences, nothing is recorded."""
    if not os.path.exists("expected_output.txt"):
        print("✗ אין קובץ expected_output.txt - חזרו לחלק א' (golden_snapshot)")
        return False
    golden_text = open("expected_output.txt", encoding="utf-8").read()
    try:
        out = run_report(parts)
    except NotImplementedError:
        raise
    except Exception as e:
        print(f"✗ אדום - התוכנית קרסה ({type(e).__name__}: {e})")
        print("  ← חוזרים צעד אחורה: הצעד לא נרשם")
        return False
    if out == golden_text:
        COMMITS[message] = True
        if not quiet:
            print(f"✓ ירוק - נרשם commit: {message}")
        return True
    diff = [l for l in difflib.unified_diff(golden_text.splitlines(), out.splitlines(), "golden", "now", lineterm="", n=0)
            if not l.startswith(("---", "+++", "@@"))]
    print("✗ אדום - הפלט השתנה. ההפרשים הראשונים:")
    for l in diff[:6]:
        print("   ", l)
    print("  ← חוזרים צעד אחורה: הצעד לא נרשם")
    return False


def _tree(func):
    try:
        return ast.parse(textwrap.dedent(inspect.getsource(func)))
    except (OSError, TypeError, SyntaxError):
        return None


def _short_names(func, min_len=3):
    t = _tree(func)
    if t is None:
        return None
    names = {n.id for n in ast.walk(t) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}
    names |= {a.arg for a in ast.walk(t) if isinstance(a, ast.arg)}
    return sorted(n for n in names if len(n) < min_len)


def _called_names(func):
    t = _tree(func)
    if t is None:
        return None
    return {n.func.id for n in ast.walk(t) if isinstance(n, ast.Call) and isinstance(n.func, ast.Name)}


def _loaded_names(func):
    t = _tree(func)
    if t is None:
        return None
    return {n.id for n in ast.walk(t) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)}


def _int_constants(func):
    t = _tree(func)
    if t is None:
        return None
    return sorted({n.value for n in ast.walk(t) if isinstance(n, ast.Constant) and type(n.value) is int})


print("✓ המנוע מוכן")

✓ המנוע מוכן


<div dir="rtl">

## שלב 0 - נקודת פתיחה: ירוק (בלי לשנות כלום)

לפני שנוגעים בקוד: ה"commit" הראשון. התוכנית המקורית מול ה-golden:

</div>

In [12]:
refactor_step_check("Start: messy report")

✓ ירוק - נרשם commit: Start: messy report


True

<div dir="rtl">

## איך נראה אדום ומה עושים

"מישהו" (או כלי AI) ביקש "לתקן" משהו שנראה לו כמו באג, ושינה **מספר אחד** בפונקציה: `3 * x["d"]` ← `2 * x["d"]`. נבדוק אם זה עדיין Refactoring:

</div>

<div dir="rtl">

**לפני שמריצים:** האם הבדיקה תישאר ירוקה אחרי שינוי של ספרה אחת? ומה לדעתכם יופיע בהודעת האדום?

**התחזית שלי:** אדומה. מספר ימי הביטחון משפיע על נקודת ההזמנה, ולכן ישתנה מספר ההתראות ואולי גם הרשימה. ההודעה תציג את השורות הראשונות שבהן הפלט שונה מה-golden.

</div>

In [13]:
def low_stock_oops(o, p, s, st):
    d = o.groupby("product_id")["quantity"].sum() / 365
    x = p.merge(s, on="supplier_id").merge(st, on="product_id")
    x["d"] = x["product_id"].map(d).fillna(0)
    x["rop"] = x["d"] * x["lead_time_days"] + 2 * x["d"]      # <-- was 3
    a = x[x["on_hand"] < x["rop"]].copy()
    a["gap"] = a["rop"] - a["on_hand"]
    return a.sort_values("gap", ascending=False)

refactor_step_check("Oops: 'fix' safety days", alerts=low_stock_oops)

✗ אדום - הפלט השתנה. ההפרשים הראשונים:
    -LOW STOCK ALERTS: 57
    -P039 Office Chair Mat Pro 1 12.16
    -P154 Plant Pot Plus 5 12.25
    -P017 Desk Organizer Basic 12 17.28
    -P088 Water Bottle Mini 1 6.19
    -P160 Bath Towel Mini 0 3.31
  ← חוזרים צעד אחורה: הצעד לא נרשם


False

<div dir="rtl">

**(א)** הבדיקה אדומה. לפי הכלל מההרצאה - מה עושים עכשיו, ולמה **לא** מבקשים מהכלי "לתקן את האדום"? (רמז: מה יהיה בידיים שלכם אחרי כמה "תיקונים" כאלה?)

**תשובה:** חוזרים צעד אחורה (ל-commit הירוק האחרון) ומנסים שוב צעד קטן יותר. תיקון על גבי תקלה מערבב שתי בעיות: כבר לא ברור מה גרם למה. המנוע עושה את זה בדיוק: הצעד האדום לא נרשם, והגרסה הקודמת נשארת. וגם: שינוי הספרה היה שינוי **התנהגות** ולא Refactoring.

</div>

<div dir="rtl">

## שלב 1 - שמות ברורים (חובה)

הריח הראשון: **שמות לא ברורים** (`o, p, s, st, d, x, a`). ממשו `low_stock_v1(orders, products, suppliers, stock)` - **אותה פונקציה בדיוק**, רק עם שמות שמסבירים (`daily_demand` במקום `d`, `merged` במקום `x`, `alerts` במקום `a`...). **צעד אחד בלבד: שמות. לא לגעת במספרים ולא בלוגיקה.**

**שימו לב (זה ה-API של הפונקציה, בהרצאה: "ההבטחה שלה"):** שאר התוכנית קוראת מהטבלה שהיא מחזירה את העמודות `product_id`, `name_x`, `on_hand`, `rop`. אל תשנו את שמותיהן, וגם לא את סדר השורות (ממוין לפי `gap` מהגבוה לנמוך). את שם `name_x` יוצר `merge` בעצמו.

**כלים:** העתיקו את `low_stock_messy` ושנו שמות של משתנים ופרמטרים. כל שם צריך להיות באורך **3 תווים לפחות**.

</div>

In [14]:
def low_stock_v1(orders, products, suppliers, stock):
    """Same behaviour as low_stock_messy, but with clear variable names (every name at least 3 letters).
    Returns the DataFrame of products below their reorder point, sorted by gap (high to low),
    with the columns product_id, name_x, on_hand, rop (unchanged names!).
    """
    daily_demand = orders.groupby("product_id")["quantity"].sum() / 365
    merged = products.merge(suppliers, on="supplier_id").merge(stock, on="product_id")
    merged["daily_demand"] = merged["product_id"].map(daily_demand).fillna(0)
    merged["rop"] = merged["daily_demand"] * merged["lead_time_days"] + 3 * merged["daily_demand"]
    alerts = merged[merged["on_hand"] < merged["rop"]].copy()
    alerts["gap"] = alerts["rop"] - alerts["on_hand"]
    return alerts.sort_values("gap", ascending=False)

In [15]:
# בדיקה - חלק ב', שלב 1
def check_b1():
    short = _short_names(low_stock_v1)
    called = _called_names(low_stock_v1)
    assert called is None or "low_stock_messy" not in called, "low_stock_v1: כתבו את הפונקציה מחדש עם שמות ברורים, לא קוראים ל-low_stock_messy"
    assert not short, f"low_stock_v1: עדיין יש שמות קצרים מדי (פחות מ-3 תווים): {short}"
    assert refactor_step_check("Step 1: clear names in low_stock", quiet=True, alerts=low_stock_v1), \
        "low_stock_v1: ה-golden אדום - חזרו לגרסה הקודמת ושנו רק שמות (לא מספרים ולא עמודות מוחזרות)"
    return True

check_b1() and print("✓ חלק ב' שלב 1: שמות ברורים, golden ירוק - נרשם commit")

✓ חלק ב' שלב 1: שמות ברורים, golden ירוק - נרשם commit


<div dir="rtl">

## שלב 2 - חילוץ פונקציות (חובה)

הריח השני: **פונקציה ארוכה שעושה הכל.** בקובץ המקורי שני חלקי הדוח הראשונים (הכנסות לפי קטגוריה, 5 מוצרים מובילים) מחושבים בתוך `main`. חלצו אותם לשתי פונקציות. **הן רק מחזירות ערך - לא מדפיסות** (ההדפסה היא עניין של הממשק; יותר על זה בשאלה שאחרי הבדיקה):

- `revenue_by_category(orders, products)` ← `Series`: סך ההכנסה לכל `category`, ממוין מהגבוה לנמוך. (צריך `merge` של `orders` עם העמודות `product_id` ו-`category` של `products`.)
- `top_products(orders, n=5)` ← `Series`: סך ההכנסה לכל `product_name`, ממוין מהגבוה לנמוך, רק `n` הראשונים.

הכנסה = `quantity * unit_price`. (**אל תשנו את `orders` המקורית**: עבדו על `orders.copy()`.)

**כלים:** `groupby(...)["rev"].sum()`, `sort_values(ascending=False)`, `.head(n)`, `df.merge(other[["product_id", "category"]], on="product_id")`.

</div>

In [16]:
def revenue_by_category(orders, products):
    """Returns a Series: index = category, value = total revenue (quantity * unit_price), sorted high to low.
    Does not print. e.g. two categories X (revenue 24) and Y (revenue 5) -> X first, then Y.
    """
    o = orders.copy()
    o["rev"] = o["quantity"] * o["unit_price"]
    merged = o.merge(products[["product_id", "category"]], on="product_id")
    return merged.groupby("category")["rev"].sum().sort_values(ascending=False)


def top_products(orders, n=5):
    """Returns a Series: index = product_name, value = total revenue, sorted high to low, only the first n.
    Does not print.
    """
    o = orders.copy()
    o["rev"] = o["quantity"] * o["unit_price"]
    return o.groupby("product_name")["rev"].sum().sort_values(ascending=False).head(n)

In [17]:
# בדיקה - חלק ב', שלב 2
tiny_orders = pd.DataFrame({"product_id": ["P1", "P2", "P1"], "product_name": ["a", "b", "a"],
                            "quantity": [2, 1, 1], "unit_price": [10.0, 5.0, 4.0]})
tiny_products = pd.DataFrame({"product_id": ["P1", "P2"], "category": ["X", "Y"]})

def check_b2():
    r = revenue_by_category(tiny_orders, tiny_products)
    assert list(r.index) == ["X", "Y"] and list(r.values) == [24.0, 5.0], "revenue_by_category: בדוגמה הקטנה X=24 ו-Y=5, ממוינים מהגבוה לנמוך"
    t = top_products(tiny_orders, 1)
    assert list(t.index) == ["a"] and list(t.values) == [24.0], "top_products: בדוגמה הקטנה a=24 (20+4), ורק n=1 מוצר"
    assert "rev" not in tiny_orders.columns, "אסור לשנות את orders המקורית - עבדו על orders.copy()"
    assert refactor_step_check("Step 2: extract revenue_by_category and top_products", quiet=True,
                               alerts=low_stock_v1, by_category=revenue_by_category, top=top_products), \
        "ה-golden אדום - חזרו צעד אחורה. הפונקציות צריכות להחזיר Series ממוין מהגבוה לנמוך, בלי להדפיס"
    return True

check_b2() and print("✓ חלק ב' שלב 2: שתי פונקציות חולצו, golden ירוק - נרשם commit")

✓ חלק ב' שלב 2: שתי פונקציות חולצו, golden ירוק - נרשם commit


<div dir="rtl">

**(ב)** הפונקציות ששלחתם **מחזירות** ערך, וההדפסה נעשית במקום אחר (במנוע). לאיזו **שכבה** (ממשק / לוגיקה / נתונים) שייכת כל אחת מהן? ולמה הבדיקה הקטנה בשלב 2 (שלוש שורות נתונים) הייתה אפשרית רק כי הן לא מדפיסות ולא קוראות קובץ?

**תשובה:** שתיהן שכבת **לוגיקה** (חישוב כללים עסקיים); ההדפסה היא ממשק; קריאת ה-CSV היא נתונים. פונקציה שמקבלת נתונים ומחזירה תוצאה אפשר לבדוק בשורה אחת על נתוני דוגמה. אילו היתה גם קוראת קובץ ומדפיסה, הבדיקה הייתה צריכה קובץ אמיתי ולקרוא פלט מהמסך.

</div>

<div dir="rtl">

## שלב 3 - הסרת שכפול (חובה)

הריח השלישי: **שכפול.** בשתי הפונקציות משלב 2 מופיעה אותה שורה: `o["rev"] = o["quantity"] * o["unit_price"]`. אם נחליט שהכנסה כוללת גם הנחה - נצטרך לזכור לשנות בשני מקומות (וב-AI, שרואה פונקציה אחת בכל פעם, זה בדיוק איך נוצרות סתירות).

1. ממשו `add_revenue(orders)`: מחזירה **עותק** של הטבלה עם עמודה חדשה `revenue` = `quantity * unit_price`.
2. כתבו מחדש את שתי הפונקציות כ-`revenue_by_category_v2` ו-`top_products_v2`, כך שיקראו ל-`add_revenue` במקום לחשב בעצמן.

</div>

In [18]:
def add_revenue(orders):
    """Returns a COPY of orders with a new column 'revenue' = quantity * unit_price (orders itself is not changed)."""
    out = orders.copy()
    out["revenue"] = out["quantity"] * out["unit_price"]
    return out


def revenue_by_category_v2(orders, products):
    """Same result as revenue_by_category, but it calls add_revenue (no duplicated formula)."""
    merged = add_revenue(orders).merge(products[["product_id", "category"]], on="product_id")
    return merged.groupby("category")["revenue"].sum().sort_values(ascending=False)


def top_products_v2(orders, n=5):
    """Same result as top_products, but it calls add_revenue (no duplicated formula)."""
    return add_revenue(orders).groupby("product_name")["revenue"].sum().sort_values(ascending=False).head(n)

In [19]:
# בדיקה - חלק ב', שלב 3
def check_b3():
    r = add_revenue(tiny_orders)
    assert list(r["revenue"]) == [20.0, 5.0, 4.0], "add_revenue: revenue = quantity * unit_price בכל שורה"
    assert "revenue" not in tiny_orders.columns, "add_revenue: אסור לשנות את הטבלה המקורית - החזירו עותק (copy)"
    for f in (revenue_by_category_v2, top_products_v2):
        called = _called_names(f)
        assert called is None or "add_revenue" in called, f"{f.__name__}: צריך לקרוא ל-add_revenue במקום לחשב את ההכנסה בעצמה"
    assert list(revenue_by_category_v2(tiny_orders, tiny_products).values) == [24.0, 5.0], "revenue_by_category_v2: התוצאה צריכה להישאר זהה (X=24, Y=5)"
    assert list(top_products_v2(tiny_orders, 1).values) == [24.0], "top_products_v2: התוצאה צריכה להישאר זהה (a=24)"
    assert refactor_step_check("Step 3: remove duplicated revenue formula", quiet=True, alerts=low_stock_v1,
                               by_category=revenue_by_category_v2, top=top_products_v2), \
        "ה-golden אדום - חזרו צעד אחורה (לגרסאות משלב 2) ונסו שוב"
    return True

check_b3() and print("✓ חלק ב' שלב 3: השכפול הוסר, golden ירוק - נרשם commit")

✓ חלק ב' שלב 3: השכפול הוסר, golden ירוק - נרשם commit


<div dir="rtl">

## שלב 4 - מספרי קסם הופכים לקבועים (מומלץ)

הריח הרביעי: **מספר קסם.** מה זה `3` ב-`+ 3 * x["d"]`? ימי ביטחון? כמות? אף אחד לא יודע בלי לנחש. ומה זה `365`?

הגדירו שני קבועים (שמות בדיוק כך, באותיות גדולות): `SAFETY_DAYS` (ימי ביטחון) ו-`DAYS_IN_YEAR` (ימים בשנה), וכתבו `low_stock_v2` - כמו `low_stock_v1` אבל **בלי המספרים 3 ו-365 בתוך הפונקציה**, רק עם השמות. (הערך `0` ב-`fillna(0)` הוא בסדר.)

</div>

In [20]:
SAFETY_DAYS = 3
DAYS_IN_YEAR = 365


def low_stock_v2(orders, products, suppliers, stock):
    """Same behaviour as low_stock_v1, but the numbers 3 and 365 come from the constants
    SAFETY_DAYS and DAYS_IN_YEAR (the function must not contain 3 or 365 itself)."""
    daily_demand = orders.groupby("product_id")["quantity"].sum() / DAYS_IN_YEAR
    merged = products.merge(suppliers, on="supplier_id").merge(stock, on="product_id")
    merged["daily_demand"] = merged["product_id"].map(daily_demand).fillna(0)
    merged["rop"] = merged["daily_demand"] * merged["lead_time_days"] + SAFETY_DAYS * merged["daily_demand"]
    alerts = merged[merged["on_hand"] < merged["rop"]].copy()
    alerts["gap"] = alerts["rop"] - alerts["on_hand"]
    return alerts.sort_values("gap", ascending=False)

In [21]:
# בדיקה - חלק ב', שלב 4
def check_b4():
    assert SAFETY_DAYS == 3, "SAFETY_DAYS צריך להיות 3 (כמו במספר הקסם המקורי)"
    assert DAYS_IN_YEAR == 365, "DAYS_IN_YEAR צריך להיות 365"
    consts = _int_constants(low_stock_v2)
    assert consts is None or (3 not in consts and 365 not in consts), "low_stock_v2: עדיין יש 3 או 365 בתוך הפונקציה - השתמשו בשמות הקבועים"
    used = _loaded_names(low_stock_v2)
    assert used is None or {"SAFETY_DAYS", "DAYS_IN_YEAR"} <= used, "low_stock_v2: הפונקציה צריכה להשתמש ב-SAFETY_DAYS וב-DAYS_IN_YEAR"
    o, p, s, st = _load()
    n_normal = len(low_stock_v2(o, p, s, st))
    old = globals()["SAFETY_DAYS"]
    try:
        globals()["SAFETY_DAYS"] = 0
        n_zero = len(low_stock_v2(o, p, s, st))
    finally:
        globals()["SAFETY_DAYS"] = old
    assert n_zero < n_normal, "low_stock_v2: שינוי SAFETY_DAYS צריך לשנות את התוצאה - כנראה הערך מוטמע בפונקציה"
    assert refactor_step_check("Step 4: named constants SAFETY_DAYS, DAYS_IN_YEAR", quiet=True, alerts=low_stock_v2,
                               by_category=revenue_by_category_v2, top=top_products_v2), \
        "ה-golden אדום - חזרו צעד אחורה. הערכים של הקבועים חייבים להיות 3 ו-365"
    return True

check_b4() and print("✓ חלק ב' שלב 4: הקבועים בשימוש, golden ירוק - נרשם commit")

✓ חלק ב' שלב 4: הקבועים בשימוש, golden ירוק - נרשם commit


<div dir="rtl">

## שלב 5 - צעד אמיתי: מציעים ל-AI, מחליטים לפי ה-golden (מומלץ)

עכשיו אותו מעגל, אבל **הצעד נכתב על ידי כלי AI**. המטרה: להחליט אם **לקבל** את ההצעה, לפי ה-golden - לא לפי "נראה טוב".

1. פתחו את עוזר ה-AI של Colab (כפתור ✨ **Gemini**, או "Generate" בתא קוד) - או כל כלי צ'אט. (מותר להדביק רק את הקוד שלמטה: זה קוד הקורס, לא מידע רגיש.)
2. העתיקו אליו את הפרומפט הבא **כמו שהוא** והדביקו אחריו את הפונקציה `low_stock_messy` (ממש מהתא למעלה):

```
Refactor ONLY the following Python function: give it clear variable names, add a short
docstring and replace magic numbers with named constants. Do not change what it returns:
keep the same arguments, the same returned columns (product_id, name_x, on_hand, rop)
and the same row order. Show only the code.
```

3. הדביקו את מה שקיבלתם בתא הבא **תחת השם `my_ai_alerts`** (אם הכלי קרא לה אחרת - שנו את השם), והריצו. בתא שאחריו תראו **diff** (מה השתנה לעומת המקור): קראו אותו, כמו שמלמדת ההרצאה.
4. הריצו את ה-golden (התא הבא) וכתבו החלטה: `"accepted"` רק אם הוא ירוק, אחרת `"rejected"`.

</div>

In [22]:
# (הפתרון: דוגמה לקוד שכלי AI עשוי להחזיר)
def my_ai_alerts(orders, products, suppliers, stock):
    """Return products whose stock is below the reorder point, largest gap first."""
    safety_days = 3
    days_per_year = 365
    daily_demand = orders.groupby("product_id")["quantity"].sum() / days_per_year
    merged = products.merge(suppliers, on="supplier_id").merge(stock, on="product_id")
    merged["daily_demand"] = merged["product_id"].map(daily_demand).fillna(0)
    merged["rop"] = merged["daily_demand"] * merged["lead_time_days"] + safety_days * merged["daily_demand"]
    alerts = merged[merged["on_hand"] < merged["rop"]].copy()
    alerts["gap"] = alerts["rop"] - alerts["on_hand"]
    return alerts.sort_values("gap", ascending=False)

In [23]:
# diff: מה השתנה לעומת הפונקציה המקורית? (קראו לפני שמריצים את ה-golden)
try:
    before = inspect.getsource(low_stock_messy).splitlines()
    after = inspect.getsource(my_ai_alerts).splitlines()
    for line in difflib.unified_diff(before, after, "original", "ai", lineterm=""):
        print(line)
except (OSError, TypeError):
    print("(לא הצלחנו להציג diff אוטומטי - השוו את שתי הפונקציות בעיניים)")

--- original
+++ ai
@@ -1,9 +1,11 @@
-def low_stock_messy(o, p, s, st):
-    """The low-stock block of messy_report.py, extracted as is (names unchanged)."""
-    d = o.groupby("product_id")["quantity"].sum() / 365
-    x = p.merge(s, on="supplier_id").merge(st, on="product_id")
-    x["d"] = x["product_id"].map(d).fillna(0)
-    x["rop"] = x["d"] * x["lead_time_days"] + 3 * x["d"]
-    a = x[x["on_hand"] < x["rop"]].copy()
-    a["gap"] = a["rop"] - a["on_hand"]
-    return a.sort_values("gap", ascending=False)
+def my_ai_alerts(orders, products, suppliers, stock):
+    """Return products whose stock is below the reorder point, largest gap first."""
+    safety_days = 3
+    days_per_year = 365
+    daily_demand = orders.groupby("product_id")["quantity"].sum() / days_per_year
+    merged = products.merge(suppliers, on="supplier_id").merge(stock, on="product_id")
+    merged["daily_demand"] = merged["product_id"].map(daily_demand).fillna(0)
+    merged["rop"] = merged["daily_demand"] *

In [24]:
refactor_step_check("Step 5: AI refactor of low_stock", alerts=my_ai_alerts)

✓ ירוק - נרשם commit: Step 5: AI refactor of low_stock


True

In [25]:
ai_decision = "accepted"
ai_note = "הכלי שינה שמות והוסיף docstring, ואת המספרים 3 ו-365 החליף במשתנים מקומיים; לא שינה את מה שהפונקציה מחזירה."

In [26]:
# בדיקה - חלק ב', שלב 5
def check_b5():
    green = refactor_step_check("Step 5: AI refactor of low_stock", quiet=True, alerts=my_ai_alerts)
    assert ai_decision in ("accepted", "rejected"), 'ai_decision: כתבו "accepted" או "rejected"'
    expected = "accepted" if green else "rejected"
    assert ai_decision == expected, f'ai_decision: ה-golden {"ירוק" if green else "אדום"}, ולכן ההחלטה הנכונה היא "{expected}"'
    assert isinstance(ai_note, str) and len(ai_note.split()) >= 4 and not ai_note.startswith("("), "ai_note: כתבו משפט שמתאר מה הכלי שינה"
    return True

check_b5() and print("✓ חלק ב' שלב 5: ההחלטה מתאימה לתוצאת ה-golden")

✓ חלק ב' שלב 5: ההחלטה מתאימה לתוצאת ה-golden


<div dir="rtl">

**(ג)** מה הכלי שינה **מעבר למה שביקשתם** (או: מה ההבדל בין ה-diff לבין הפלט)? ואם ה-golden היה ירוק - האם זה מוכיח שההצעה טובה? מה עוד הייתם בודקים לפני שמקבלים?

**תשובה:** הכלי הוסיף שמות ו-docstring ושינה את הנוסחה לצורה מקבילה. ירוק מוכיח רק שהפלט של הדוח זהה, בדיוק על הנתונים האלה: לא שההצעה יפה, ולא שהיא נכונה בקצוות (למשל מוצר בלי מכירות). לכן קוראים את ה-diff ובודקים גם שמות, תיעוד ונקודות קצה, ורק אז עושים commit.

</div>

<div dir="rtl">

## סיכום החלק: כמה commits קיבלתם?

התא מריץ את התוכנית שהורכבה מהגרסאות הסופיות שלכם מול ה-golden, ומציג את רשימת ה-commits שנרשמו (כל צעד ירוק = commit):

</div>

In [27]:
FINAL_PARTS = dict(alerts=low_stock_v2, by_category=revenue_by_category_v2, top=top_products_v2)
refactor_step_check("Final: all parts together", **FINAL_PARTS)
print(len(COMMITS), "commits:")
for message in COMMITS:
    print("  -", message)

✓ ירוק - נרשם commit: Final: all parts together
7 commits:
  - Start: messy report
  - Step 1: clear names in low_stock
  - Step 2: extract revenue_by_category and top_products
  - Step 3: remove duplicated revenue formula
  - Step 4: named constants SAFETY_DAYS, DAYS_IN_YEAR
  - Step 5: AI refactor of low_stock
  - Final: all parts together


<div dir="rtl">

---

# חלק ג' - קנה מידה: מודדים, לא מנחשים (8 דק')

## הרקע

"זה עובד על 3,000 שורות" לא אומר "זה יעבוד על 300,000". בהרצאה ראינו לולאה על שורות שהאטה פי 10 כשהנתונים גדלו פי 10 (וכאן נראה משהו גרוע יותר). **הכלל: מודדים לפני שמייעלים** (ולא מייעלים כל דבר). וצריך לוודא שהגרסה המהירה נותנת **אותה תשובה**: ייעול הוא סוג של Refactoring, והצעד שבודק אותו הוא אותו golden.

## המשימה

לכל לקוח - סך הכמויות שהזמין. הקלט: רשימה של זוגות `(לקוח, כמות)`, והפלט: מילון `{לקוח: סך כמויות}`. הנה נתונים סינתטיים (כ־43% ממספר השורות בקירוב הם לקוחות שונים) וגרסה "איטית" שכתב מישהו:

</div>

In [28]:
def make_rows(n):
    """n synthetic order lines (customer, quantity); about 0.43*n different customers."""
    rnd = random.Random(42)
    return [(f"C{rnd.randrange(n // 2):05d}", rnd.randint(1, 9)) for _ in range(n)]


def totals_slow(rows):
    """Total quantity per customer. For every row: search the list of known customers."""
    keys, sums = [], []
    for customer, qty in rows:
        if customer in keys:
            sums[keys.index(customer)] += qty
        else:
            keys.append(customer)
            sums.append(qty)
    return dict(zip(keys, sums))


print(make_rows(6))
print(totals_slow(make_rows(6)))

[('C00002', 2), ('C00000', 5), ('C00000', 4), ('C00000', 2), ('C00002', 9), ('C00000', 7)]
{'C00002': 11, 'C00000': 18}


<div dir="rtl">

## שלב 1 - שעון עצר (חובה)

ממשו `measure(func, rows, repeat=1)`: מריצה `func(rows)` בסך הכל `repeat` פעמים, ומחזירה את **הזמן הקצר ביותר** (בשניות) מבין ההרצאות.

**כלים:** `time.perf_counter()` מחזיר שעון מדויק בשניות. מודדים כך: `start = time.perf_counter()`, מריצים, ואז `time.perf_counter() - start`. את הקטן מכמה מספרים נותן `min(רשימה)`.

</div>

In [29]:
def measure(func, rows, repeat=1):
    """Runs func(rows) `repeat` times and returns the smallest running time in seconds (use time.perf_counter()).
    e.g. measure(totals_slow, make_rows(1000), repeat=3) -> 0.004 (depends on the computer)
    """
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        func(rows)
        times.append(time.perf_counter() - start)
    return min(times)

In [30]:
# בדיקה - חלק ג', שלב 1
def check_c1():
    t = measure(lambda rows: time.sleep(0.05), [])
    assert 0.04 <= t < 0.3, f"measure: פונקציה שישנה 0.05 שניות צריכה להימדד כ-0.05 בערך, התקבל {t:.3f}"
    pauses = iter([0.15, 0.05, 0.10])
    t3 = measure(lambda rows: time.sleep(next(pauses)), [], repeat=3)
    assert 0.04 <= t3 < 0.09, f"measure: עם repeat=3 צריך להחזיר את הזמן הקצר ביותר (0.05), התקבל {t3:.3f}"
    return True

check_c1() and print("✓ חלק ג' שלב 1: measure עובדת")

✓ חלק ג' שלב 1: measure עובדת


<div dir="rtl">

## שלב 2 - הגרסה המהירה (חובה)

ממשו `totals_fast(rows)`: אותה תוצאה כמו `totals_slow`, אבל עם **מילון** (`dict`) שמצטבר תוך כדי מעבר **אחד** על השורות. בחיפוש במילון לא עוברים על כל המפתחות: הוא קופץ ישר אל המפתח.

**כלים:** `totals = {}`, ובכל שורה: `totals[customer] = totals.get(customer, 0) + qty`. (`dict.get(מפתח, ברירת_מחדל)` מחזיר את הערך, או את ברירת המחדל אם המפתח עוד לא קיים.)

</div>

In [31]:
def totals_fast(rows):
    """Same result as totals_slow (a dict {customer: total quantity}), using a dict and one pass over the rows.
    e.g. totals_fast([("A", 2), ("B", 1), ("A", 3)]) -> {"A": 5, "B": 1}
    """
    totals = {}
    for customer, qty in rows:
        totals[customer] = totals.get(customer, 0) + qty
    return totals

In [32]:
# בדיקה - חלק ג', שלב 2
def check_c2():
    assert totals_fast([("A", 2), ("B", 1), ("A", 3)]) == {"A": 5, "B": 1}, "totals_fast: הדוגמה מה-docstring (A=5, B=1)"
    assert totals_fast([]) == {}, "totals_fast: על רשימה ריקה צריך להחזיר מילון ריק"
    rows = make_rows(500)
    assert totals_fast(rows) == totals_slow(rows), "totals_fast: התוצאה שונה מ-totals_slow על 500 שורות"
    big = make_rows(3_000)
    assert measure(totals_slow, big) > 5 * measure(totals_fast, big), "totals_fast: צריכה להיות מהירה בהרבה מ-totals_slow (פי 5 לפחות) כבר על 3,000 שורות"
    return True

check_c2() and print("✓ חלק ג' שלב 2: totals_fast נכונה ומהירה")

✓ חלק ג' שלב 2: totals_fast נכונה ומהירה


<div dir="rtl">

## שלב 3 - אותה תשובה? (חובה)

לפני שמחליפים גרסה איטית בגרסה מהירה: **בודקים שהתשובה זהה.** זה ה-golden של הייעול. ממשו `same_answer(f, g, rows)`: מחזירה `True` אם `f(rows)` ו-`g(rows)` נותנות אותה תוצאה בדיוק.

</div>

In [33]:
def same_answer(f, g, rows):
    """True if f(rows) and g(rows) return exactly the same result (the golden test of an optimisation)."""
    return f(rows) == g(rows)

In [34]:
# בדיקה - חלק ג', שלב 3
def check_c3():
    def buggy(rows):          # "optimisation" that drops the last row
        return totals_fast(rows[:-1])
    rows = make_rows(300)
    assert same_answer(totals_slow, totals_fast, rows) is True, "same_answer: שתי הגרסאות זהות, צריך להחזיר True"
    assert same_answer(totals_slow, buggy, rows) is False, "same_answer: גרסה שמפילה את השורה האחרונה צריכה להיחשב שונה (False)"
    return True

check_c3() and print("✓ חלק ג' שלב 3: same_answer תופסת 'ייעול' שגוי")

✓ חלק ג' שלב 3: same_answer תופסת 'ייעול' שגוי


<div dir="rtl">

## שלב 4 - מדידה אמיתית: 1,000 מול 10,000 שורות

</div>

<div dir="rtl">

**לפני שמריצים:** הגרסה האיטית עוברת בכל שורה על רשימת הלקוחות הידועים. אם ב-1,000 שורות היא לוקחת T, כמה פעמים **יותר** תיקח ב-10,000 שורות (פי 10? פי 100? פי 1,000?)? ומה לגבי הגרסה המהירה?

**התחזית שלי:** האיטית: פי 100 בערך (או קרוב לזה), כי גם מספר השורות גדל פי 10 וגם רשימת הלקוחות שבה מחפשים גדלה פי 10 (n × n). המהירה: פי 10 בערך, כי עוברת פעם אחת על כל שורה.

</div>

In [35]:
sizes = [1_000, 10_000]
results = {}
for n in sizes:
    rows = make_rows(n)
    assert same_answer(totals_slow, totals_fast, rows), "התוצאות שונות - לא משווים זמנים של גרסאות שונות"
    for name, func in [("slow", totals_slow), ("fast", totals_fast)]:
        results[(name, n)] = measure(func, rows, repeat=3 if n <= 1_000 else 1)
        print(f"{n:>7,} שורות   {name}: {results[(name, n)]:9.5f} שניות")

  1,000 שורות   slow:   0.00338 שניות
  1,000 שורות   fast:   0.00006 שניות


 10,000 שורות   slow:   0.25562 שניות
 10,000 שורות   fast:   0.00063 שניות


<div dir="rtl">

חשבו את **גורם הגידול** של כל גרסה: הזמן ב-10,000 שורות לחלק בזמן ב-1,000 שורות (`results[("slow", 10_000)] / results[("slow", 1_000)]`).

</div>

In [36]:
slow_growth = results[("slow", 10_000)] / results[("slow", 1_000)]
fast_growth = results[("fast", 10_000)] / results[("fast", 1_000)]
print("slow גדלה פי", round(slow_growth), "| fast גדלה פי", round(fast_growth))

slow גדלה פי 76 | fast גדלה פי 10


In [37]:
# בדיקה - חלק ג', שלב 4
def check_c4():
    assert isinstance(slow_growth, (int, float)) and isinstance(fast_growth, (int, float)), "חשבו את שני גורמי הגידול (מספרים)"
    assert slow_growth > 20, f"slow_growth={slow_growth:.1f}: האיטית צריכה לגדול הרבה יותר מפי 10. בדקו שחילקתם את הזמן ב-10,000 בזמן ב-1,000"
    assert slow_growth > fast_growth, "slow_growth צריך להיות גדול מ-fast_growth"
    return True

check_c4() and print("✓ חלק ג' שלב 4: גורמי הגידול מחושבים")

✓ חלק ג' שלב 4: גורמי הגידול מחושבים


<div dir="rtl">

**(א)** הנתונים גדלו פי 10 והזמן של `totals_slow` גדל הרבה יותר מפי 10. הסבירו **למה** (מה הקוד עושה בכל שורה?). ואם 10,000 שורות לקחו X שניות - מה תחזית גסה ל-100,000 שורות (בלי להריץ)?

**תשובה:** בכל שורה `customer in keys` ו-`keys.index(...)` עוברים על רשימת הלקוחות הידועים, והיא גדלה יחד עם הנתונים: מספר הפעולות גדל כמו n × n, כלומר פי 100 כשהנתונים גדלים פי 10. מ-10,000 ל-100,000 נצפה לפי 100 נוספים, כלומר מאות שניות (דקות) במקום חלקיק שנייה בגרסה עם המילון.

</div>

<div dir="rtl">

**(ב)** בהרצאה: "האם צריך תמיד לייעל?" - לא, קודם מודדים. באיזה מצב **לא** הייתם טורחים להחליף את `totals_slow`, ולמה `same_answer` הכרחי לפני שמחליפים?

**תשובה:** אם הנתונים תמיד קטנים (אלפי שורות) והקוד רץ מילי-שניות, הייעול לא שווה את המורכבות: קודם מודדים ורק אם יש בעיה מייעלים. same_answer הכרחי כי ייעול משנה קוד, ו-Refactoring בלי הוכחה שהתוצאה זהה עלול להחליף תשובה נכונה באחרת (בדיוק כמו golden בחלק א').

</div>

<div dir="rtl">

---

# חלק ד' - קליטת קוד זר ומסמך העברה (11 דק')

## הרקע

בהרצאה: **חמישה צעדי קליטה** - README והרצה, מיפוי מבנה, נקודת כניסה, זרימה אחת, מסמך העברה. והכלל: **קודם מריצים, ורק אחר כך קוראים.** קיבלתם את `legacy-repo`: כלי שנותן ציון לספקים. כל מה שיש ל-README שלו להגיד:

</div>

In [38]:
def load_repo(folder):
    """Reads the .py and .md files of a folder into a dict {file name: text}."""
    files = {}
    for name in sorted(os.listdir(folder)):
        if name.endswith((".py", ".md")):
            files[name] = open(os.path.join(folder, name), encoding="utf-8").read()
    return files


def run_repo(args=(), cwd="legacy-repo", script="main.py"):
    """Runs `python script args` inside the folder cwd; returns what it printed (or the last error line)."""
    result = subprocess.run([sys.executable, script, *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        return "FAILED: " + result.stderr.strip().splitlines()[-1]
    return result.stdout


LEGACY = load_repo("legacy-repo")
print(list(LEGACY))
print(LEGACY["README.md"])

['README.md', 'config.py', 'loader.py', 'main.py', 'scoring.py']
# scorecard
Run main.py



<div dir="rtl">

## שלב 0 - קודם מריצים (בלי להבין עדיין)

ה-README אומר רק "Run main.py". נסו להריץ, עם ובלי ארגומנט (הארגומנט הוא ציון מינימלי, ראו ב-`main.py`), ו**גם מהתיקייה הלא נכונה**:

</div>

<div dir="rtl">

**לפני שמריצים:** (1) מה ידפיס `python main.py 0.7`? כמה ספקים ייכנסו לטבלה? (2) מה יקרה אם נריץ את אותו קובץ **מתיקייה אחרת**?

**התחזית שלי:** (1) טבלה של שם ספק וציון, ממוינת מהגבוה לנמוך, רק ספקים עם ציון 0.7 ומעלה: 4 ספקים (C, A, E, G). (2) כנראה שגיאת 'קובץ לא נמצא', כי הנתיב ל-suppliers.csv יחסי לתיקיית ההרצה ולא לתיקיית הקובץ.

</div>

In [39]:
print("--- python main.py 0.7 ---")
print(run_repo(["0.7"]))
print("--- python main.py (בלי ארגומנט) ---")
print(run_repo())
print("--- אותו קובץ, אבל הרצה מתיקייה אחרת (elsewhere) ---")
os.makedirs("elsewhere", exist_ok=True)
print(run_repo(cwd="elsewhere", script="../legacy-repo/main.py"))

--- python main.py 0.7 ---


      name  score
Supplier C  0.913
Supplier A  0.823
Supplier E  0.762
Supplier G  0.762

--- python main.py (בלי ארגומנט) ---


      name  score
Supplier C  0.913
Supplier A  0.823
Supplier E  0.762
Supplier G  0.762
Supplier F  0.697
Supplier B  0.667
Supplier D  0.661
Supplier H  0.631

--- אותו קובץ, אבל הרצה מתיקייה אחרת (elsewhere) ---


FAILED: FileNotFoundError: [Errno 2] No such file or directory: './suppliers.csv'


<div dir="rtl">

## שלב 1 - מיפוי: מה יש במאגר? (חובה)

ממשו `inventory_repo(files)`: מקבלת מילון `{שם קובץ: טקסט}` ומחזירה מילון עם ארבעה מפתחות:

- `"py_files"` - רשימה **ממוינת** של שמות קבצי `.py`.
- `"functions"` - מילון: לכל קובץ `.py` רשימת שמות הפונקציות שמוגדרות בו ברמה העליונה (שורה שמתחילה ב-`def `).
- `"untested"` - רשימה (ממוינת) של קבצי `.py` **ללא בדיקות**: קובץ שאינו קובץ בדיקה עצמו (שם שמתחיל ב-`test_`) ואין במאגר קובץ בשם `test_<שם הקובץ>`. (למשל ל-`util.py` יש בדיקות אם קיים `test_util.py`.)
- `"todos"` - רשימה של `(שם_קובץ, מספר_שורה, טקסט_השורה)` לכל שורה (בכל קובץ) שמכילה `TODO` או `FIXME`. מספור שורות מ-1, והטקסט בלי רווחים בקצוות.

**כלים:** `for name in sorted(files):`, `files[name].splitlines()`, `for i, line in enumerate(lines, 1):`. שם פונקציה: אם `line.startswith("def ")` - הטקסט בין `"def "` לבין `"("`: `line[4:line.index("(")]`. `name.endswith(".py")`, `name.startswith("test_")`.

</div>

In [40]:
def inventory_repo(files):
    """files: {file name: text}. Returns a dict:
    {"py_files": [sorted .py names],
     "functions": {py file: [top-level function names]},
     "untested": [sorted .py files that are not tests and have no test_<name> file],
     "todos": [(file, line number from 1, stripped line) for every line with TODO or FIXME]}
    """
    py_files = sorted(name for name in files if name.endswith(".py"))
    functions = {}
    for name in py_files:
        functions[name] = [line[4:line.index("(")] for line in files[name].splitlines() if line.startswith("def ")]
    untested = [name for name in py_files if not name.startswith("test_") and "test_" + name not in py_files]
    todos = []
    for name in sorted(files):
        for number, line in enumerate(files[name].splitlines(), 1):
            if "TODO" in line or "FIXME" in line:
                todos.append((name, number, line.strip()))
    return {"py_files": py_files, "functions": functions, "untested": untested, "todos": todos}

In [41]:
# בדיקה - חלק ד', שלב 1
tiny_repo = {
    "app.py": "import x\n\ndef start():\n    # TODO: handle errors\n    pass\n\ndef stop():\n    pass\n",
    "util.py": "def helper(a):\n    return a  # FIXME later\n",
    "test_util.py": "def test_helper():\n    assert True\n",
    "README.md": "# tiny\nTODO write docs\n",
}

def check_d1():
    info = inventory_repo(tiny_repo)
    assert sorted(info["py_files"]) == ["app.py", "test_util.py", "util.py"], "inventory_repo: py_files - רק קבצי .py (בלי README.md)"
    assert info["functions"] == {"app.py": ["start", "stop"], "test_util.py": ["test_helper"], "util.py": ["helper"]}, \
        "inventory_repo: functions - לכל קובץ .py רשימת הפונקציות שלו (שורות שמתחילות ב-def)"
    assert sorted(info["untested"]) == ["app.py"], "inventory_repo: untested - util.py נבדק (יש test_util.py), test_util.py הוא בדיקה בעצמו, ונשאר רק app.py"
    assert sorted(tuple(t) for t in info["todos"]) == [("README.md", 2, "TODO write docs"), ("app.py", 4, "# TODO: handle errors"),
                                                       ("util.py", 2, "return a  # FIXME later")], \
        "inventory_repo: todos - (שם קובץ, מספר שורה מ-1, השורה בלי רווחים בקצוות) לכל TODO או FIXME, גם בקבצי md"
    real = inventory_repo(LEGACY)
    assert real["functions"]["main.py"] == ["run"] and real["functions"]["scoring.py"] == ["score"], "inventory_repo: על legacy-repo צריך למצוא את run ו-score"
    assert sorted(real["untested"]) == ["config.py", "loader.py", "main.py", "scoring.py"], "inventory_repo: ב-legacy-repo אין בדיקות בכלל"
    return True

check_d1() and print("✓ חלק ד' שלב 1: inventory_repo עובדת")
print(inventory_repo(LEGACY))

✓ חלק ד' שלב 1: inventory_repo עובדת
{'py_files': ['config.py', 'loader.py', 'main.py', 'scoring.py'], 'functions': {'config.py': [], 'loader.py': ['load_suppliers'], 'main.py': ['run'], 'scoring.py': ['score']}, 'untested': ['config.py', 'loader.py', 'main.py', 'scoring.py'], 'todos': []}


<div dir="rtl">

**(א)** מה **גילה** המיפוי של `legacy-repo` שה-README (שתי שורות) לא סיפר? קשרו לשלב ההרצה: מה למדתם מההרצה מתיקייה אחרת?

**תשובה:** במאגר ארבעה קבצי קוד (main הוא ה-run, loader קורא את הנתונים, scoring נותן ציון, config מחזיק הגדרות), אין אף קובץ בדיקות ואין TODO שמסביר משהו. ההרצה מתיקייה אחרת נכשלה, כי DATA_DIR='.' הוא יחסי לתיקיית ההרצה ולא לקובץ: נקודת סיכון שאין לה זכר ב-README.

</div>

<div dir="rtl">

## שלב 2 - בונים את מסמך ההעברה (חובה)

ממשו `write_handoff(info, what, run_cmd, broken, risks, questions, path="HANDOFF.md")`. הפונקציה בונה מסמך Markdown **לפי התבנית** (שורת הכותרת הראשית ואז שישה חלקים בדיוק בסדר הזה), כותבת אותו לקובץ `path` ומחזירה את הטקסט:

```
# מסמך העברה

## מה זה עושה
<what>

## איך מריצים
```
<run_cmd>
```

## מבנה
- `<קובץ>`: <שמות הפונקציות, מופרדות בפסיק, או (אין פונקציות)>
  (שורה כזו לכל קובץ ב-info["py_files"])

ללא בדיקות: <קבצים מ-info["untested"] מופרדים בפסיק, או: אין>
TODO פתוחים: <מספר הפריטים ב-info["todos"]>

## מה שבור או חסר
- <פריט מתוך broken>   (אם הרשימה ריקה: שורה אחת: - (אין))

## נקודות סיכון
1. <סיכון ראשון>
2. <סיכון שני>
3. <סיכון שלישי>

## שאלות פתוחות
- <שאלה מתוך questions>   (אם ריקה: - (אין))
```

**כלל API (שקף "עיצוב API של פונקציה"):** `risks` חייבת להכיל **בדיוק 3** פריטים. אחרת הפונקציה **נכשלת בקול**: `raise ValueError("...")` עם הסבר, ולא מחזירה מסמך חלקי בשקט.

**כלים:** בונים רשימה `lines`, מוסיפים אליה בעזרת `lines.append(...)` ו-`lines.extend(...)`, ובסוף `"\n".join(lines)`. מספור: `for i, risk in enumerate(risks, 1): f"{i}. {risk}"`. כתיבה לקובץ: `open(path, "w", encoding="utf-8")`.

</div>

In [42]:
def write_handoff(info, what, run_cmd, broken, risks, questions, path="HANDOFF.md"):
    """Builds the handoff document (markdown, template above), writes it to `path` and returns the text.
    Raises ValueError if risks does not contain exactly 3 items.
    """
    if len(risks) != 3:
        raise ValueError(f"write_handoff: need exactly 3 risks, got {len(risks)}")
    lines = ["# מסמך העברה", "", "## מה זה עושה", what, "", "## איך מריצים", "```", run_cmd, "```", "", "## מבנה"]
    for name in info["py_files"]:
        funcs = ", ".join(info["functions"][name]) or "(אין פונקציות)"
        lines.append(f"- `{name}`: {funcs}")
    lines += ["", "ללא בדיקות: " + (", ".join(info["untested"]) or "אין"), f"TODO פתוחים: {len(info['todos'])}", ""]
    lines += ["## מה שבור או חסר"] + ([f"- {item}" for item in broken] or ["- (אין)"]) + [""]
    lines += ["## נקודות סיכון"] + [f"{i}. {risk}" for i, risk in enumerate(risks, 1)] + [""]
    lines += ["## שאלות פתוחות"] + ([f"- {q}" for q in questions] or ["- (אין)"])
    text = "\n".join(lines) + "\n"
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return text

In [43]:
# בדיקה - חלק ד', שלב 2
def _section(doc, title):
    lines, take, out = doc.splitlines(), False, []
    for line in lines:
        if line.startswith("## "):
            take = (line == "## " + title)
        elif take:
            out.append(line)
    return "\n".join(out)

def check_d2():
    info = {"py_files": ["a.py", "b.py"], "functions": {"a.py": ["f", "g"], "b.py": []},
            "untested": ["a.py"], "todos": [("a.py", 3, "TODO x")]}
    doc = write_handoff(info, "what text", "python a.py 5", ["b1", "b2"], ["r1", "r2", "r3"], ["q1"], "_HANDOFF_test.md")
    assert open("_HANDOFF_test.md", encoding="utf-8").read() == doc, "write_handoff: צריך לכתוב לקובץ את אותו טקסט שמוחזר"
    assert doc.splitlines()[0].startswith("# "), "write_handoff: השורה הראשונה היא כותרת ראשית (# מסמך העברה)"
    headers = [l for l in doc.splitlines() if l.startswith("## ")]
    assert headers == ["## מה זה עושה", "## איך מריצים", "## מבנה", "## מה שבור או חסר", "## נקודות סיכון", "## שאלות פתוחות"], \
        f"write_handoff: ששת החלקים בדיוק בסדר מהתבנית. נמצאו: {headers}"
    assert "what text" in _section(doc, "מה זה עושה"), "write_handoff: בחלק 'מה זה עושה' צריך להופיע הטקסט של what"
    run = _section(doc, "איך מריצים")
    assert "```" in run and "python a.py 5" in run, "write_handoff: הפקודה צריכה להופיע בתוך בלוק קוד (``` לפני ואחרי)"
    struct = _section(doc, "מבנה")
    assert "`a.py`" in struct and "f, g" in struct and "`b.py`" in struct, "write_handoff: בחלק 'מבנה' שורה לכל קובץ: `שם`: פונקציות, מופרדות בפסיק"
    assert "(אין פונקציות)" in struct, "write_handoff: לקובץ בלי פונקציות כתבו (אין פונקציות)"
    assert "ללא בדיקות: a.py" in struct and "TODO פתוחים: 1" in struct, "write_handoff: חסרות השורות 'ללא בדיקות: ...' ו-'TODO פתוחים: N'"
    broken = _section(doc, "מה שבור או חסר")
    assert "- b1" in broken and "- b2" in broken, "write_handoff: כל פריט מ-broken בשורה שמתחילה ב- '- '"
    risks = [l for l in _section(doc, "נקודות סיכון").splitlines() if l.strip()]
    assert risks == ["1. r1", "2. r2", "3. r3"], "write_handoff: שלושת הסיכונים ממוספרים 1. 2. 3."
    assert "- q1" in _section(doc, "שאלות פתוחות"), "write_handoff: כל שאלה בשורה שמתחילה ב- '- '"
    empty = write_handoff(info, "w", "python a.py", [], ["r1", "r2", "r3"], [], "_HANDOFF_test.md")
    assert "- (אין)" in _section(empty, "מה שבור או חסר"), "write_handoff: אם broken ריקה, כתבו שורה '- (אין)'"
    try:
        write_handoff(info, "w", "python a.py", [], ["r1", "r2"], [], "_HANDOFF_test.md")
    except ValueError:
        pass
    else:
        raise AssertionError("write_handoff: עם 2 סיכונים בלבד צריך להרים ValueError (להיכשל בקול) ולא לבנות מסמך חלקי")
    return True

check_d2() and print("✓ חלק ד' שלב 2: write_handoff בונה מסמך לפי התבנית")

✓ חלק ד' שלב 2: write_handoff בונה מסמך לפי התבנית


<div dir="rtl">

## שלב 3 - מסמך אמיתי ל-`legacy-repo` (מומלץ)

עכשיו מה שדורש **הבנה ולא קוד**. מלאו את המשתנים לפי מה ש**הרצתם וראיתם** בשלב 0 ובמיפוי (לא לפי ניחוש, ולא לפי מה שכלי AI "חושב"):

- `my_what` - משפט אחד: מה הכלי עושה, כולל איך מחושב הציון (עיינו ב-`scoring.py` וב-`config.py` בעזרת `print(LEGACY["scoring.py"])`).
- `my_run` - פקודה שמריצים **מתוך תיקיית `legacy-repo`**, שמתחילה ב-`python`, למשל `python main.py 0.5`. **התא הבודק יריץ אותה** כדי לוודא שהיא עובדת.
- `my_broken` - רשימה של מה שבור או חסר (לפחות פריט אחד).
- `my_risks` - **בדיוק 3** נקודות סיכון (כל אחת לפחות 15 תווים, שונות זו מזו).
- `my_questions` - לפחות 3 שאלות לכותבים המקוריים (שאלה חזקה: "למה משקל זמן האספקה הוא 0.4?"; חלשה: "מה זה עושה?").

</div>

In [44]:
my_what = "כלי שקורא את suppliers.csv ומדפיס טבלת ספקים ממוינת לפי ציון = 0.6 × שיעור אספקה בזמן + 0.4 × (1 − זמן אספקה/21, כשזמן האספקה מוגבל ל-21 ימים); ארגומנט אופציונלי הוא ציון מינימלי."
my_run = "python main.py 0.7"
my_broken = ["ה-README כמעט ריק: שתי שורות, בלי הארגומנט, התלויות או הפלט",
             "אין אף בדיקה אוטומטית",
             "הרצה מתיקייה אחרת נכשלת ב-FileNotFoundError על suppliers.csv"]
my_risks = ["DATA_DIR הוא '.' - הנתיב יחסי לתיקיית ההרצה ולא לתיקיית הקוד",
            "אין בדיקות, ולכן אי אפשר לשנות את הנוסחה בלי חשש לשבור משהו",
            "אין טיפול בקובץ חסר או בערך חסר ב-on_time_rate, והמספרים 0.6, 0.4 ו-21 הם הנחות בלי הסבר"]
my_questions = ["למה המשקלים הם 0.6 ו-0.4?",
                "מה קורה אם on_time_rate חסר בשורה?",
                "האם 21 ימים הוא מקסימום עסקי, או רק ברירת מחדל?"]

In [45]:
handoff_doc = write_handoff(inventory_repo(LEGACY), my_what, my_run, my_broken, my_risks, my_questions, "HANDOFF.md")
print(handoff_doc)

# מסמך העברה

## מה זה עושה
כלי שקורא את suppliers.csv ומדפיס טבלת ספקים ממוינת לפי ציון = 0.6 × שיעור אספקה בזמן + 0.4 × (1 − זמן אספקה/21, כשזמן האספקה מוגבל ל-21 ימים); ארגומנט אופציונלי הוא ציון מינימלי.

## איך מריצים
```
python main.py 0.7
```

## מבנה
- `config.py`: (אין פונקציות)
- `loader.py`: load_suppliers
- `main.py`: run
- `scoring.py`: score

ללא בדיקות: config.py, loader.py, main.py, scoring.py
TODO פתוחים: 0

## מה שבור או חסר
- ה-README כמעט ריק: שתי שורות, בלי הארגומנט, התלויות או הפלט
- אין אף בדיקה אוטומטית
- הרצה מתיקייה אחרת נכשלת ב-FileNotFoundError על suppliers.csv

## נקודות סיכון
1. DATA_DIR הוא '.' - הנתיב יחסי לתיקיית ההרצה ולא לתיקיית הקוד
2. אין בדיקות, ולכן אי אפשר לשנות את הנוסחה בלי חשש לשבור משהו
3. אין טיפול בקובץ חסר או בערך חסר ב-on_time_rate, והמספרים 0.6, 0.4 ו-21 הם הנחות בלי הסבר

## שאלות פתוחות
- למה המשקלים הם 0.6 ו-0.4?
- מה קורה אם on_time_rate חסר בשורה?
- האם 21 ימים הוא מקסימום עסקי, או רק ברירת מחדל?



In [46]:
# בדיקה - חלק ד', שלב 3
def check_d3():
    assert not my_what.startswith("(") and len(my_what.split()) >= 6, "my_what: כתבו משפט אחד של לפחות 6 מילים"
    assert my_run.startswith("python "), "my_run: הפקודה צריכה להתחיל ב-python (ולרוץ מתוך legacy-repo)"
    import shlex
    result = subprocess.run([sys.executable] + shlex.split(my_run)[1:], cwd="legacy-repo", capture_output=True, text=True)
    assert result.returncode == 0, "my_run: הפקודה נכשלה כשהורצה מתוך legacy-repo. הריצו אותה בעצמכם כמו בשלב 0. השגיאה: " + result.stderr.strip()[-120:]
    assert "main.py" in my_run, "my_run: הפקודה צריכה להריץ את main.py"
    assert len(my_broken) >= 1, "my_broken: לפחות פריט אחד (למשל משהו שראיתם בהרצה או במיפוי)"
    assert len(my_risks) == 3 and len({r.strip() for r in my_risks}) == 3, "my_risks: בדיוק 3 סיכונים שונים"
    assert all(len(r) >= 15 for r in my_risks), "my_risks: כל סיכון צריך להיות משפט (לפחות 15 תווים)"
    assert len(my_questions) >= 3, "my_questions: לפחות 3 שאלות לכותבים"
    doc = globals().get("handoff_doc") or ""
    assert "(כתבו" not in doc and doc, "handoff_doc: נשארו תחליפים - מלאו את כל החלקים והריצו שוב את התא שבונה את המסמך"
    for item in my_risks:
        assert item in doc, "handoff_doc: הריצו מחדש את התא שבונה את המסמך אחרי שעדכנתם את הרשימות"
    return True

check_d3() and print("✓ חלק ד' שלב 3: מסמך ההעברה מלא, והפקודה שבו עובדת")

✓ חלק ד' שלב 3: מסמך ההעברה מלא, והפקודה שבו עובדת


<div dir="rtl">

**(ב)** נניח שהייתם מבקשים מכלי AI **לכתוב** את מסמך ההעברה הזה (מהקבצים בלבד, בלי הרצה). אילו חלקים בו הוא יכול לנסח טוב, ואילו חלקים **רק אתם** יכולים לכתוב? ומה הכלל שמונע מכם לשלוח מסמך שגוי?

**תשובה:** כלי יכול לנסח טוב את 'מה זה עושה', 'איך מריצים' ו'מבנה' (קבצים ותפקידים) כי הם נגזרים מהקוד. אבל הוא עלול להמציא התנהגות שלא קיימת (הזיה), ולכן מריצים ובודקים. 'מה שבור' ו'נקודות סיכון' (למשל שהנתיב נשבר מתיקייה אחרת) נמצאים בהרצה ולא בקוד, והחלטות וסיכונים בפרויקט אמיתי יכול לכתוב רק מי שהיה שם. הכלל: קודם מריצים, אחר כך כותבים.

</div>

<div dir="rtl">

---

# חלק ה' - מסקנה (2 דק')

ארבעה משפטים, אחד לכל חלק. כל משפט בנוי כך: **מה מימשתם ומדדתם ← איזו חוליה/עיקרון זה מראה ← מה עושים בעבודה עם AI.**

1. **רשת ביטחון:** כתבתי golden_snapshot ו-is_green ומצאתי ששינוי שם נשאר ירוק ושינוי ספרה אדום, ולכן לפני שמבקשים מ-AI לסדר קוד נועלים את הפלט הנוכחי.
2. **Refactoring בצעדים:** סידרתי את הדוח בארבעה צעדים קטנים כשכל צעד נרשם רק אחרי golden ירוק, ולכן כשהצעד של הכלי יוצא אדום חוזרים צעד אחורה, ומבקשים בקשה קטנה וממוקדת במקום 'לשכתב הכל'.
3. **קנה מידה:** מדדתי שהגרסה האיטית גדלה בערך פי 50-100 כשהנתונים גדלו פי 10 והמילון רק פי 10, ולכן מודדים לפני ואחרי ובודקים שהתשובה זהה, גם כשהכלי כתב את הלולאה.
4. **קליטת קוד זר:** הרצה ומיפוי של legacy-repo חשפו נתיב שביר ואין בדיקות שה-README לא סיפר, ולכן קודם מריצים ואז כותבים מסמך העברה ובודקים כל טענה שכלי AI טוען.

</div>

<div dir="rtl">

---

# בונוס

**1. להפריד לוגיקה מהדפסה.** כתבו `count_alerts(orders, products, suppliers, stock)` שמחזירה **מספר** (כמה התראות מלאי נמוך), בלי להדפיס, ובדקו אותה על 3-4 שורות נתונים שבניתם בעצמכם (בלי קבצים). מה זה מלמד על שכבת הלוגיקה?

**2. pandas.** חשבו את אותו סכום לכל לקוח עם `pd.DataFrame(make_rows(100_000), columns=["customer", "qty"]).groupby("customer")["qty"].sum()` ומדדו אותו מול `totals_fast`. מי מהיר יותר ב-100,000 שורות? (אל תריצו את `totals_slow` על 100,000: זה ייקח דקות.)

**3. הפרויקט שלכם.** הכניסו לתוך מילון `{שם קובץ: טקסט}` שניים-שלושה קבצים מהפרויקט הקבוצתי והריצו עליהם `inventory_repo`. כמה קבצים ללא בדיקות? מה היה כדאי להוסיף ל-README?

</div>

In [47]:
# תא לבונוס 1

In [48]:
# תא לבונוס 2

In [49]:
# תא לבונוס 3

<div dir="rtl">

---

# בדיקה מסכמת - הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime ← Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓` (שלבי "מומלץ" אינם חובה).

</div>

In [50]:
for name, check in [("א' - golden_snapshot", check_a1),
                    ("א' - is_green", check_a2),
                    ("ב' - שלב 1: שמות ברורים", check_b1),
                    ("ב' - שלב 2: חילוץ פונקציות", check_b2),
                    ("ב' - שלב 3: הסרת שכפול", check_b3),
                    ("ב' - שלב 4: קבועים (מומלץ)", check_b4),
                    ("ב' - שלב 5: צעד AI (מומלץ)", check_b5),
                    ("ג' - measure", check_c1),
                    ("ג' - totals_fast", check_c2),
                    ("ג' - same_answer", check_c3),
                    ("ג' - גורמי גידול", check_c4),
                    ("ד' - inventory_repo", check_d1),
                    ("ד' - write_handoff", check_d2),
                    ("ד' - מסמך ל-legacy-repo (מומלץ)", check_d3)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")

✓ א' - golden_snapshot


✓ א' - is_green
✓ ב' - שלב 1: שמות ברורים
✓ ב' - שלב 2: חילוץ פונקציות
✓ ב' - שלב 3: הסרת שכפול
✓ ב' - שלב 4: קבועים (מומלץ)
✓ ב' - שלב 5: צעד AI (מומלץ)


✓ ג' - measure
✓ ג' - totals_fast
✓ ג' - same_answer
✓ ג' - גורמי גידול
✓ ד' - inventory_repo
✓ ד' - write_handoff


✓ ד' - מסמך ל-legacy-repo (מומלץ)
